# Onboarding: physics-constrained solar flare & SEP forecasting

This notebook is the orientation for the `SolarFlareForcast` project. Read it top to bottom once; every code cell is
light (seconds) and nothing here trains a model.

| Part | What you get |
|---|---|
| 1. Why this matters | Space weather, who uses these forecasts, NOAA R and S scales |
| 2. The data | GOES XRS soft X-rays and ≥10 MeV protons: what they measure, units, physics, real vs synthetic |
| 3. The ML problem | Sliding windows, persistence baseline, leakage-free splits |
| 4. The physics loss | Each penalty, where it comes from, and a toy demo of what it punishes |
| 5. Results so far | Tables and figures from `sweeps/` |
| 6. Kinks to work out | Honest limitations, ranked |
| 7. Future work | Projects that need your decisions, and topics I can walk you through |
| 8. Glossary | |

Run the setup cell first. It activates this project's environment, the same pattern as your other projects.

In [ ]:
include("sol_setup.jl")
using CSV
show_png(path) = isdefined(Main, :IJulia) ? display("image/png", read(path)) : println("[figure] ", path)

## 1. Why this matters

**Space weather** is the set of conditions on the Sun and in the solar wind that affect technology and people.
Two of the most direct hazards begin at the Sun within minutes to hours:

* **Solar flares.** Magnetic energy released in an active region heats coronal plasma to 10–30 MK. The X-ray burst
  reaches Earth in about 8 minutes and ionises the dayside D-layer of the ionosphere. That causes **HF radio blackouts**
  (aviation and maritime comms, over-the-horizon radar) and GNSS degradation.
* **Solar energetic particle (SEP) events.** Protons and ions accelerated to MeV–GeV energies by flares and by
  CME-driven shocks. They arrive in tens of minutes to hours. They cause **radiation dose** to astronauts (especially
  beyond low Earth orbit, e.g. Artemis), single-event upsets and solar-panel degradation on satellites,
  polar-cap absorption, and radiation exposure on polar flight routes.

NOAA's Space Weather Prediction Center (SWPC) issues alerts on two scales that map directly onto the two channels
this project forecasts:

| Radio blackout | GOES XRS-B peak | Radiation storm | ≥10 MeV protons |
|---|---|---|---|
| R1 minor | M1 (10⁻⁵ W m⁻²) | S1 minor | 10 pfu |
| R2 moderate | M5 | S2 moderate | 10² pfu |
| R3 strong | X1 (10⁻⁴) | S3 strong | 10³ pfu |
| R4 severe | X10 | S4 severe | 10⁴ pfu |
| R5 extreme | X20 | S5 extreme | 10⁵ pfu |

**Who cares (and who hires):** SWPC forecasters; NASA's Space Radiation Analysis Group (astronaut dose); satellite
operators; national-lab space-science groups that build and analyse particle instruments (e.g. LANL's energetic-particle
sensors flown on GPS satellites) and that need people comfortable with both instrument data and ML. The skill this
project shows, *ML that respects instrument physics and is evaluated honestly against an operational baseline*,
is the skill those groups look for.

## 2. The data

### 2a. GOES XRS: soft X-ray flux

The GOES-R series (GOES-16 to -19) carries the **X-Ray Sensor (XRS)**, with two broadband channels:

* **XRS-A** 0.05–0.4 nm (harder, more temperature-sensitive)
* **XRS-B** 0.1–0.8 nm. *This channel defines the flare class* (A, B, C, M, X = 10⁻⁸ … 10⁻⁴ W m⁻²; the number after the
  letter is a multiplier, so M2.3 = 2.3 × 10⁻⁵ W m⁻²).

Units are **W m⁻²** integrated over the band, at Earth. SWPC serves 1-minute averages. The emission is mostly
**thermal bremsstrahlung and lines from hot flare-loop plasma**, so the light curve is a thermodynamic record:

* **Impulsive phase** (minutes): electron beams deposit energy and chromospheric plasma "evaporates" into the loops.
  Flux rises fast. The **Neupert effect** says the soft X-ray flux tracks the *time integral* of the non-thermal
  energy input, so the soft X-ray *derivative* looks like the hard X-ray burst. That is why the model gets the
  1-minute XRS increment as an extra input channel.
* **Gradual phase** (tens of minutes to hours): the loops cool, first by thermal conduction and then by radiation.
  The flux decays roughly exponentially. It **cannot** drop faster than the fastest cooling time allows, which is
  the physics behind the decay-bound penalty in part 4.

The background between flares (the B–C level "quiet Sun") is set by how many active regions are on the disk. It
varies over hours to days, with a 27-day modulation from solar rotation.

Flare sizes follow a **power law**, dN/dF ∝ F^−α with α ≈ 1.8–2: many small flares, very few big ones. That makes
this a rare-event problem.

The cell below generates the project's synthetic 90-day telemetry and zooms in on its largest flare.

In [ ]:
df, events = generate_telemetry(TelemetryConfig(seed = 42))
cls = first.(flare_class.(events.peak_flux))
println(nrow(df), " one-minute samples; ", nrow(events), " flares: ",
        join(["$(c)=$(count(==(c), cls))" for c in "BCMX"], ", "), "; ", count(events.sep), " SEP events")

k = argmax(events.peak_flux); ipk = events.peak_index[k]
r = max(1, ipk - 90):min(nrow(df), ipk + 360)
fig = Figure(size = (900, 330))
ax = Axis(fig[1, 1]; yscale = log10, xlabel = "Minutes from peak", ylabel = "XRS-B (W m⁻²)",
          title = "Largest synthetic flare: $(flare_class(events.peak_flux[k])), rise τ = $(round(events.rise_min[k]; digits=1)) min, decay τ = $(round(events.decay_min[k]; digits=1)) min")
lines!(ax, r .- ipk, df.xray[r]; color = "#2a78d6", label = "observed (2% noise)")
lines!(ax, r .- ipk, df.xray_clean[r]; color = :black, linestyle = :dot, label = "noise-free truth")
axislegend(ax; position = :rt)
fig

**Check the power law yourself.** On a log–log plot of the cumulative number of flares above a threshold, the slope is
−(α − 1) ≈ −0.9. A slope that changes with your threshold choice would be evidence of incompleteness (small flares
hidden under the background), a real issue with GOES catalogs.

In [ ]:
F = sort(events.peak_flux; rev = true)
N = 1:length(F)
sel = F .>= 1e-6                                     # fit above C1, where the catalog is complete
A = hcat(ones(count(sel)), log10.(F[sel]))
b = A \ log10.(N[sel])
fig = Figure(size = (600, 360))
ax = Axis(fig[1, 1]; xscale = log10, yscale = log10, xlabel = "Peak flux F (W m⁻²)", ylabel = "N(> F)",
          title = "Cumulative flare-size distribution, fitted slope $(round(b[2]; digits = 2)) ⇒ α ≈ $(round(1 - b[2]; digits = 2))")
scatter!(ax, F, N; markersize = 4, color = "#2a78d6")
lines!(ax, F[sel], 10 .^ (A * b); color = :black, linewidth = 2)
fig

### 2b. ≥10 MeV integral proton flux: the SEP channel

GOES-R measures protons with **SGPS** (Solar and Galactic Proton Sensor, part of the SEISS suite). SWPC publishes
*integral* fluxes, the number of protons above an energy threshold, in **pfu = particles cm⁻² s⁻¹ sr⁻¹**.
The ≥10 MeV channel defines the S scale. The quiet level is about 0.1–1 pfu (galactic cosmic rays plus instrument
background).

SEP physics in one paragraph: particles are accelerated in the flare reconnection region (*impulsive* events: short,
electron- and ³He-rich) and, for the biggest events, at the shock driven by a fast CME (*gradual* events: long and
proton-rich). They must then travel along the Parker-spiral magnetic field to Earth, so events from the Sun's
**western hemisphere** (magnetically connected) arrive faster and stronger. The fastest particles arrive first
(**velocity dispersion**), so onsets lag the flare by tens of minutes and the flux then rises for hours and decays
over a day or more as particles escape. That slow decay is why the proton decay bound in part 4 uses a
much longer timescale than the X-ray one.

The synthetic generator has the main ingredients (only ≥M1 flares can launch SEPs, probability and intensity grow with
flare size, transport delay, hours-long rise, day-long decay), but **not** connectivity or CMEs. More on that in
part 6.

In [ ]:
sep = events[events.sep, :]
println("SEP events (flare class → peak pfu):")
for e in eachrow(sep)
    println("  ", rpad(flare_class(e.peak_flux), 6), " → ", round(e.sep_peak_pfu; digits = 1), " pfu; flare peak → SEP peak ",
            round((e.sep_peak_index - e.peak_index) / 60; digits = 1), " h")
end
e = sep[argmax(sep.sep_peak_pfu), :]
r = max(1, e.peak_index - 120):min(nrow(df), e.peak_index + 48 * 60)
t = (r .- e.peak_index) ./ 60
fig = Figure(size = (900, 460))
ax1 = Axis(fig[1, 1]; yscale = log10, ylabel = "XRS-B (W m⁻²)", title = "Flare → SEP: $(flare_class(e.peak_flux)) flare", xticklabelsvisible = false)
ax2 = Axis(fig[2, 1]; yscale = log10, ylabel = "≥10 MeV (pfu)", xlabel = "Hours from flare peak")
linkxaxes!(ax1, ax2)
lines!(ax1, t, df.xray[r]; color = "#2a78d6")
lines!(ax2, t, df.proton[r]; color = "#eb6834")
hlines!(ax2, SEP_THRESHOLD_PFU; color = :gray, linestyle = :dash)
text!(ax2, t[end], SEP_THRESHOLD_PFU; text = "S1", align = (:right, :bottom), color = :gray)
fig

### 2c. Real data from NOAA SWPC and NCEI

`load_swpc()` pulls SWPC's live 7-day JSON products (`xrays-7-day.json`, `integral-protons-7-day.json`) and puts them on
the **same schema** as the synthetic data (`time, minute, xray, proton`). Everything downstream works unchanged.
Every download is cached in `data/raw/` with a UTC timestamp, and `load_swpc_cache()` merges all cached downloads.
For multi-year training sets the source is NOAA's science-quality NCEI GOES-R archive (netCDF): `load_ncei` reads it
into the same schema (§5 has the details and the validation against SWPC).

Things real data has that the synthetic data doesn't. Spot them in the plot:
* **Data gaps** and dropped rows (the windowing code refuses to build a window across a gap).
* **Eclipse season** near the equinoxes. The satellite passes through Earth's shadow around local midnight, which
  causes dips or dropouts in the X-ray record.
* **Primary/secondary satellite switches**, small calibration offsets, and particle contamination of the XRS.

In [ ]:
real = try
    load_swpc_cache()
catch err
    println("No cached SWPC data yet. Run `load_swpc()` once (needs internet).  ", err)
    nothing
end
if real !== nothing
    println(nrow(real), " real one-minute rows, ", first(real.time), " → ", last(real.time),
            "; largest flare ", flare_class(maximum(real.xray)))
    d = real.minute ./ 1440
    fig = Figure(size = (900, 420))
    ax1 = Axis(fig[1, 1]; yscale = log10, ylabel = "XRS-B (W m⁻²)", title = "Real GOES data from SWPC", xticklabelsvisible = false)
    ax2 = Axis(fig[2, 1]; yscale = log10, ylabel = "≥10 MeV (pfu)", xlabel = "Days since $(Date(first(real.time)))")
    linkxaxes!(ax1, ax2)
    lines!(ax1, d, real.xray; color = "#2a78d6", linewidth = 0.8)
    lines!(ax2, d, real.proton; color = "#eb6834", linewidth = 0.8)
    for (l, v) in pairs(XRAY_CLASS_FLOOR); hlines!(ax1, v; color = (:gray, 0.4), linestyle = :dash); end
    fig
end

## 3. The ML problem

**Task:** given the last **W = 180 min** of both channels, predict the next **H = 60 min** of both, minute by
minute, in log₁₀ flux. Log space is natural because flux spans 5+ decades and errors are multiplicative.
A 0.1 dex error is a 26% flux error at any flare size.

Design choices you should be able to defend in an interview:

1. **Persistence baseline.** "The next hour looks like now" is hard to beat at short lead times and is the standard
   operational reference. Any skill claim is *relative to persistence*.
2. **Residual architecture.** The CNN predicts the *change* from the last observation, ŷ = x_last + f(x). Its last
   layer starts at zero, so an untrained model *is* persistence, and training only adds what the data supports.
3. **Chronological split with guard gaps.** Train on days 0–63, validate on the next 13.5, test on the last 13.5. A gap of
   W + H rows between splits means no test window shares a single minute with a training window. Random splits of
   overlapping windows leak the answer and inflate scores, a very common mistake in published ML papers.
4. **Standardise with training statistics only**, for the same reason.
5. **Test set is sealed.** Weights and λ are chosen on validation. The test split is scored once.

In [ ]:
data = prepare_data(df; W = 180, H = 60)
println("train X ", size(data.train.X), "   (window minutes, channels, windows)")
println("val   X ", size(data.val.X), "   test X ", size(data.test.X))
println("standardiser μ = ", round.(data.scaler.μ; digits = 2), " (log10 units), σ = ", round.(data.scaler.σ; digits = 2))
println("rows: ", data.rows)

## 4. The physics-informed loss

Total loss = MSE + λ_decay·P_decay + λ_rough·P_rough + λ_convex·P_convex. Each penalty is computed on the forecast in
physical units (dex = log₁₀ flux), with the last observed point prepended so the jump from *now* into the forecast is
also constrained.

**P_decay, the cooling bound (one-sided).** If plasma cools with e-folding time τ, then F ∝ e^{−t/τ} and
d log₁₀F/dt = −1/(τ ln 10). With a minimum physical cooling time τ_min (5 min for X-rays, 3 h for SEP protons),
any faster drop is unphysical:

$$P_\text{decay} = \big\langle \text{ReLU}\big(-\Delta_t \log_{10}\hat F - \tfrac{1}{\tau_\min \ln 10}\big)^2 \big\rangle$$

Rises are **not** penalised. The impulsive phase really can be arbitrarily fast. This *asymmetry* is the key
difference from generic smoothing.

**P_rough, quiet-Sun curvature.** The squared second difference of the X-ray forecast, multiplied by a soft gate that
is ≈1 only when the last 30 min of input varied by less than 0.15 dex. It removes high-frequency wiggles on a quiet
background but leaves flares alone. Protons are always smooth at 1-min cadence, so their curvature is penalised
everywhere.

**P_convex, gradual-phase shape.** An exponential decay on top of a constant background is a *convex* curve in log space:
the decline slows as the flux approaches background. During a decay (input below its 60-min peak and falling), concave
bends are penalised.

**Ablation: naive smoothing.** Your original prototype penalised every first difference equally. That also flattens
genuine flare rises. It is kept as a comparison model.

The toy demo below builds four hand-made 60-min forecasts from the same quiet input and prints what each penalty says.

In [ ]:
σ = data.scaler.σ
quiet, _ = regime_gates(data.test.X, σ[1], 0.15f0)
n = findfirst(>(0.95f0), vec(quiet))                    # a quiet test window
x = data.test.X[:, :, n:n]
last_ = x[end, 1:2, 1]
H = 60
smooth = repeat(reshape(last_, 1, 2, 1), H, 1, 1)
jagged = smooth .+ 0.05f0 .* randn(Xoshiro(1), Float32, H, 2, 1)   # ±0.05σ wiggles
fastdrop = copy(smooth); fastdrop[:, 1, 1] .-= Float32.(0.4 .* (1:H) ./ σ[1])  # −0.4 dex/min: faster than any cooling
naive_risk = copy(smooth); naive_risk[20:end, 1, 1] .+= Float32(1.0 / σ[1])    # sudden 1-dex rise (a real flare could do this)
L = PhysicsLoss()
for (name, ŷ) in ("flat (persistence)" => smooth, "jagged" => jagged, "too-fast decay" => fastdrop, "sudden rise" => naive_risk)
    p = physics_terms(L, ŷ, x, σ)
    @printf("%-20s decay %.2e   rough %.2e   convex %.2e   naive %.2e\n", name, p.decay, p.rough, p.convex, p.naive)
end

Read the table row by row:
* *jagged*: `rough` (and `naive`) fire. That is the quiet-time artifact we want to kill. `decay` fires a little too, from the
  **proton** channel: its bound is tight (3-h decay time ≈ 0.0024 dex/min), so random downward wiggles break it.
* *too-fast decay*: `decay` fires hard. That's the unphysical drop.
* *sudden rise*: `decay` stays zero (rises are allowed) and `naive` fires. A one-step jump also has curvature, so
  `rough` fires a little on a quiet window, which is why its gate matters.

## 5. Results so far (sweep 1, 2026-10-18)

Full write-up: `sweeps/REPORT.md`. The cells below read the saved tables and figures, so they always show the latest run.

**Protocol recap:** λ was chosen on validation (grid: naive 0.3/3/30, physics 1e3–1e6), then the chosen models were
retrained with 3 seeds and scored once on the sealed test split against the noise-free truth.

In [ ]:
s = CSV.read(joinpath(RESULTS_DIR, "summary.csv"), DataFrame)
chosen = Dict(split(l)[1] => split(l)[2] for l in eachline(joinpath(RESULTS_DIR, "chosen.txt")))
keep = ["persistence"; [chosen[k] for k in ("mse", "naive", "phys")]]
t = filter(r -> r.tag in keep, s)
for r in eachrow(t)
    @printf("%-24s X-ray %.4f ± %.4f   active %.3f   protons %.4f   quiet roughness %.1e   TSS(≥M) %.3f\n",
            r.model, r.rmse_x_clean, coalesce(r.rmse_x_clean_sd, NaN), r.rmse_x_active, r.rmse_p_clean,
            r.rough_quiet, r.tss_m)
end

How to read it:
* **X-ray:** the physics loss gives the lowest error, about 14% below persistence and 2% below the same network trained
  on MSE alone, with almost all of the gain in **active periods**. The seed-to-seed spread is far smaller than the gap.
* **Quiet roughness:** about 77% lower than the plain CNN, which is exactly what the quiet-curvature term was for.
* **Naive smoothing:** same RMSE as the plain CNN but about 20% *less* M-class skill. It can't tell a flare rise from
  noise, which is the core argument for an *asymmetric, physics-gated* penalty.
* **Protons:** the physics model is stuck at persistence. See kink #2.

In [ ]:
show_png(joinpath(FIG_DIR, "skill_vs_horizon.png"))
show_png(joinpath(FIG_DIR, "forecast_examples.png"))

**Lead-time plot:** everything matches persistence for the first ~5 minutes (the last value is a great forecast), and
the CNNs pull ahead after that. **Examples:** in the decay panel, the plain CNN decays too slowly and the physics
model too fast at first. Both miss the true *straight-line* (pure exponential) decay in opposite directions. No model
sees the new flare in the last panel coming.

### Zero-shot on real GOES data

The synthetic-trained models are applied unchanged to 7 days of live SWPC data:

In [ ]:
z = CSV.read(joinpath(RESULTS_DIR, "zero_shot_swpc.csv"), DataFrame)
for r in eachrow(z)
    @printf("%-24s X-ray RMSE %.3f   active %.3f   quiet %.3f   protons %.3f\n", r.model, r.rmse_x, r.rmse_x_active, r.rmse_x_quiet, r.rmse_p)
end
show_png(joinpath(FIG_DIR, "swpc_zero_shot_examples.png"))

On real data the plain CNN edges out persistence (by about 2%) while the physics model is about 5% **worse**. Its
priors describe the *synthetic* flares. The real M1.4 decays faster and less convexly. The "fastest rise" panel isn't
a flare at all: it's a 1.6-dex dropout and recovery in the data, probably an eclipse-season or data-quality artefact.
This is the most important result of the night for planning: **the physics has to be validated on real light curves,
not only on the generator that encodes it.**

### Sweep 2 (2026-10-19): which term does the work, and fixing the protons

Everything below is at λ = 1e4 with one seed (sweep-1 seed spread ≈ 0.0008 dex, so smaller gaps aren't resolved).

In [ ]:
for tag in ("mse", "phys_1e4", "physp_1e4_0.01", "physp_1e4_0", "rough_1e4", "decay_1e4", "convex_1e4")
    m = filter(x -> x.tag == tag, s)
    nrow(m) == 0 && continue
    row = m[1, :]
    @printf("%-16s X-ray %.4f   protons %.4f   quiet roughness %.1e   cooling violations %.4f\n",
            tag, row.rmse_x_clean, row.rmse_p_clean, row.rough_quiet, row.decay_violation_frac)
end

* **`physp_1e4_0.01`** (proton penalties × 0.01) is best on *both* channels: protons drop below persistence again,
  and the X-ray gain is kept. This is the model carried forward to real data.
* **Quiet curvature alone** (`rough_1e4`) gives nearly the whole X-ray gain. **The cooling bound alone**
  (`decay_1e4`) matches no penalty, because it never binds. **Convexity alone** makes quiet forecasts rougher.
  So the honest headline is "a regime-gated curvature prior", not "three physics terms".

### Real data: building and validating the loader

`load_ncei(16, Date(2022,4,25), Date(2025,3,31))` downloads NOAA's science-quality GOES archive (NCEI L2, 1-min).
* **XRS-B:** the files carry per-minute quality flags (good / eclipse / bad / interpolated). Eclipse and bad minutes
  are dropped, which removes exactly the kind of dropout that fooled every model in the SWPC test above.
* **Protons:** the archive has 13 *differential* channels (1–500 MeV, two sensor heads) but no ≥10 MeV integral, so
  the loader reconstructs it: Σ flux × channel width above 10 MeV + the >500 MeV channel, averaged over both heads.
  This kind of instrument-level reasoning is what instrument groups care about.

**Validation:** XRS-B matches SWPC's real-time feed to 0.001 dex (median). The reconstructed proton peaks for NOAA's 8
official 2024 SEP events:

In [ ]:
v = CSV.read(joinpath(RESULTS_DIR, "sep_peak_validation.csv"), DataFrame)
for r in eachrow(v)
    @printf("%s   NOAA %6.0f pfu   reconstructed %6.0f pfu   ratio %.2f\n", r.listed_max, r.listed_pfu, r.recon_pfu, r.ratio)
end
@printf("median ratio %.2f, scatter %.2f dex\n", median(v.ratio), std(log10.(v.ratio)))
show_png(joinpath(FIG_DIR, "ncei_vs_swpc.png"))

Peak *times* agree to within minutes for 6 of 8 events, and all 8 cross the 10 pfu S1 threshold, as in NOAA's list.
Amplitudes are typically within a factor of ~1.7. The hardest-spectrum events read ~2× high, a known limitation
of a simple bin sum (no gap filling, no spectral fit).

### Real data: training and results (2026-10-20 → 10-29)

**Protocol:** GOES-16 NCEI L2, 1-min. Chronological split: train 2022-04 → 2023-12, validation 2024-01 → 2024-06,
test 2024-07 → 2025-03 (includes the Oct 2024 X9.0 flare and the 1810 pfu SEP event). Errors are against the
*observations* (there is no noise-free truth for real data), 3 seeds each. Four networks are compared:
* **`real_mse`:** the plain CNN, trained on real data only.
* **`real_physp_1e4_0.01`:** the sweep-2 physics loss (λ = 1e4, proton terms × 0.01), real data only.
* **`…_ft-physp_1e4_0.01_s1`:** the same physics model *pre-trained on synthetic telemetry*, then fine-tuned on real
  data (the fix proposed in kink #1).
* **`…_ft-physp_1e4_0.01_s1-pr`:** as above, but the proton half of the output layer is reset to zero before
  fine-tuning. The proton forecast restarts from persistence; the X-ray head and the shared trunk keep the prior.

In [ ]:
rs = CSV.read(joinpath(RESULTS_DIR, "real_summary.csv"), DataFrame)
for r in eachrow(rs)
    @printf("%-42s n=%d  X-ray %.4f ± %.4f   active %.3f   protons %.3f   quiet roughness %.1e   TSS(≥M) %.3f\n",
            r.config, r.n_seeds, r.rmse_x, r.rmse_x_sd, r.rmse_x_active, r.rmse_p, r.rough_quiet, r.tss_m)
end
show_png(joinpath(FIG_DIR, "real_forecast_examples.png"))

How to read it (test split, dex):
* **Training on real data fixed the zero-shot failure.** Every CNN now beats persistence (0.178), by 12–14% overall
  and by 21–23% during flares.
* **The physics loss still helps on the Sun:** 0.1552 ± 0.0006 vs 0.1569 ± 0.0006 for the plain CNN (−1.1%, about
  3× the seed spread), and quiet-time roughness is halved. That's the same direction as on synthetic data (−2.2%).
* **Synthetic pre-training helps further:** 0.1533 ± 0.0005, another −1.2% (≈3σ), and the smoothest forecasts
  (roughness 2.3e-4 vs 6.0e-4). So the generator is useful as a *prior*, even though it fails as a stand-in.
* **The cost, and the fix:** pre-training made protons slightly worse (0.122 vs 0.119), probably because the
  generator's SEP model is too simple. Resetting the proton output head before fine-tuning recovers ~40% of that
  (0.1206) and keeps the X-ray gain (0.1530 ± 0.0006). All CNNs still beat proton persistence (0.159) by ~25%.
* **Onsets:** M-class TSS differences are within about 1σ. The gain is in how flares *evolve and decay*, not in
  predicting new ones (kink #5).

**Current best model (chosen on validation):** physics CNN pre-trained on synthetic, proton head reset, fine-tuned on
GOES-16: **X-ray RMSE 0.1530 dex, −13.8% vs persistence (−22.8% during flares).** Full write-up: `sweeps/REPORT.md` (R1–R4).

### Probabilistic forecasts (2026-10-30)

A point forecast says "the flux will be X". Operational users need "what's the chance it exceeds M1?". So a small
**quantile head** was added on the frozen best model. It predicts the 5/25/50/75/95% levels of every forecast minute,
trained with the **pinball loss** (the proper score for quantiles: under-predicting the 95% level costs 0.95 per dex,
over-predicting it costs 0.05). It is compared with **static error bars**: the same quantiles of the validation errors,
fixed per lead time and regime. The question is whether the uncertainty should *depend on the situation*.

In [ ]:
ps = CSV.read(joinpath(RESULTS_DIR, "prob_summary.csv"), DataFrame)
for r in eachrow(ps)
    @printf("%-36s pinball X %.4f (active %.4f)   90%% cover %.3f   90%% width in flares %.2f dex   BSS ≥M1 %.3f   BSS ≥S1 %.3f\n",
            r.model, r.pinball_x, r.pinball_x_active, r.cover90_x, r.width90_x_active, r.bss_m, r.bss_s1)
end
show_png(joinpath(FIG_DIR, "prob_examples.png"))
show_png(joinpath(FIG_DIR, "prob_reliability.png"))

How to read it:
* **Situation-dependent uncertainty wins:** the learned head beats static error bars on every seed (pinball −2.7%),
  most of all during flares (−4%). There its 90% interval is 12% *narrower* at the same coverage: sharper and
  still honest.
* **Fan charts:** during a decay the learned band has more room below than above, as cooling physics suggests. In the
  "fastest rise" panel the flare keeps rising above even the 90% band, the same onset/peak problem as kink #5/#8.
* **Reliability diagram:** points above the diagonal mean "it happened more often than the forecast said". Every
  method under-forecasts, most of all for S1. Part of that is by construction: "max over lead times of P(exceed)" is
  a *lower bound* on the chance of exceeding at *some* time in the hour. Fix: recalibrate on validation (isotonic
  regression), or predict the event probability directly.
* **Brier skill score (BSS):** 1 is perfect, 0 is no better than always forecasting the base rate. ≥S1 skill is high
  (0.93) because proton storms last hours, so "is it already above 10 pfu?" carries most of it. For *new* M-class
  crossings the skill is only ~0.06: onsets are still the hard part.

## 6. Kinks to work out (most important first)

1. **Synthetic → real gap.** The priors (convexity, the cooling time scale) match the generator's flares, not the
   Sun's. *Largely fixed:* training on real data beats persistence, and pre-training on synthetic then fine-tuning on
   real data gives the best model (−1.4% vs real-only). Resetting the proton head before fine-tuning recovered ~40% of
   the small proton cost of pre-training; the rest sits in the shared trunk. Fitting τ_min and the convexity gate on
   real decays is still open.
2. ~~**One λ for two very different channels.**~~ *Fixed in sweep 2:* scaling the proton penalties by 0.01 stops the
   proton forecast from collapsing to persistence.
3. **Convexity is only right when the background matters.** For a big flare far above background, the log-decay is a
   straight line, and the prior bends it. Fix: gate it by the flare-to-background ratio.
4. **Real-data quality control.** Dropouts, eclipse dips, satellite switches and particle contamination look like
   flares to a model. *Partly fixed:* the NCEI loader drops eclipse- and bad-flagged minutes. Still to do: track
   primary-satellite switches, and quantify the proton reconstruction's bias on hard-spectrum events.
5. **Onsets are unforecastable from X-rays alone.** TSS on new ≥M crossings is ≈ 0.015 for every model, on synthetic
   and real data. Real flare precursors live in photospheric magnetic fields (e.g. SDO/HMI SHARP parameters), not in
   the X-ray light curve.
6. **Too few SEP events.** The real test split has only a handful of S1 events, so SEP skill numbers are anecdotal.
   Real SEP work needs decades of GOES data and event-based cross-validation.
7. ~~**Point forecasts only.**~~ *Done in P1:* a learned quantile head gives 5–95% ranges and event probabilities,
   and beats static error bars. Still open: event probabilities are biased low and need recalibration.
8. **Peak underestimation.** MSE in log space regresses toward the mean, so the impulsive-rise panel undershoots the
   peak. Quantile or asymmetric losses help here.
9. **What's not established yet:** whether the cooling bound ever binds on *real* data (it never binds on synthetic
   data), multi-seed error bars for the sweep-2 variants, and how results vary across synthetic-data seeds.
   (The per-term ablation is done: the quiet-curvature term carries the gain.)

## 7. Future work

### 7a. Projects that need your decision

Each one changes the scope or the data, so it's your call. Rough cost in brackets.

1. ~~**Train on real data (the big one).**~~ *Done (2026-10-20 → 10-24):* see §5. The physics model fine-tuned from
   synthetic is best, at −13.7% vs persistence. *Still yours to decide:* add GOES-18/19 (more years,
   satellite-to-satellite generalisation).
2. ~~**Per-channel physics weights.**~~ *Done in sweep 2:* proton penalties × 0.01 fix the proton collapse.
3. ~~**Probabilistic forecasts.**~~ *Done (P1, 2026-10-30):* quantile head on the frozen best model, scored with pinball
   loss, interval coverage, Brier skill and reliability (§5). Next: recalibrate the event probabilities on validation
   [analysis only, minutes].
4. **Two-channel thermodynamics.** Add XRS-A (0.05–0.4 nm). The A/B ratio gives plasma **temperature** and
   **emission measure** via the GOES response functions. Then constrain *physics we can see*: during cooling,
   temperature should peak before emission measure. A strong, publishable-style physics constraint.
   *Decide:* scope; it touches the generator, the model and the loss.
5. **SEP connectivity.** Add flare longitude (from the SWPC event list or HEK) as an input. Western-hemisphere flares
   are magnetically connected and produce faster, stronger SEPs. *Decide:* data source; real data only.
6. **More seeds for the sweep-2 variants.** The per-term ablation is done (the curvature term carries the gain);
   seeds 2–3 would put error bars on it. [4–6 runs × ~20 min]
7. **Target audience.** Which program or lab is this pointed at first? That decides whether the write-up leans
   instrument (calibration, data quality) or ML (architecture, uncertainty).

### 7b. Things I can walk you through (no decision needed, just ask)

* Why log space and a persistence-residual network, and why "beats persistence" is the bar.
* Why random splits of overlapping windows leak, and how the guard gap fixes it.
* Why TSS and not accuracy for rare events (an "always no flare" forecast is 99% accurate).
* How the cooling bound follows from F ∝ e^{−t/τ}, and why it's one-sided.
* Why naive smoothing hurts flare skill (it can't tell a real rise from noise).
* The Neupert effect and why the XRS derivative is a useful input feature.
* How the λ sweep was chosen on validation only, and what "sealed test set" protects against.
* How to read the skill-vs-lead-time plot (where the CNN separates from persistence, and why).
* Why pre-training on a simulator and fine-tuning on real data works (transfer learning with a physics prior).

## 8. Glossary

| Term | Meaning |
|---|---|
| **XRS-A / XRS-B** | GOES X-ray Sensor channels, 0.05–0.4 nm and 0.1–0.8 nm; XRS-B defines flare class |
| **A/B/C/M/X class** | XRS-B peak flux decades: 10⁻⁸ … 10⁻⁴ W m⁻² |
| **pfu** | particle flux unit, protons cm⁻² s⁻¹ sr⁻¹ |
| **SEP** | solar energetic particle event (here ≥10 MeV protons) |
| **S1 / R1** | NOAA radiation-storm / radio-blackout scale levels |
| **dex** | one decade in log₁₀; 0.1 dex ≈ 26% |
| **Persistence** | forecast that holds the latest value |
| **TSS** | true skill statistic = hit rate − false-alarm rate (POD − POFD); 0 = no skill, 1 = perfect |
| **Neupert effect** | soft X-ray flux ≈ time integral of non-thermal (hard X-ray) energy input |
| **Leakage** | information from the evaluation period reaching training, which inflates scores |
| **λ (lambda)** | weight of a penalty term in the loss |
| **Sealed test set** | data scored once, after every choice was made on validation |